# California House Pricing 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [3]:
# Loading the Data 
df = pd.read_csv('housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


# Understanding the Data 

In [4]:
# Shape (rows, columns)
print(f"Dataset has {df.shape[0]} Houses and {df.shape[1]} features")

# Column names
print("\nColumns:", df.columns.tolist())

# Data types
print("\nData types:")
print(df.dtypes)

# Summary statistics
df.describe()

Dataset has 20640 Houses and 10 features

Columns: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

Data types:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity        object
dtype: object


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


# Checking for the Missing Values

In [5]:
# Count missing values
missing = df.isnull().sum() # Here df is the DataFrame we are working with
print(missing[missing > 0])

# Percentage missing
missing_percent = (df.isnull().sum() / len(df)) * 100
print("\nPercentage missing:")
print(missing_percent[missing_percent > 0])

total_bedrooms    207
dtype: int64

Percentage missing:
total_bedrooms    1.002907
dtype: float64


Resolving the Missing Values

In [7]:
# We could have also added values by mean or median but since this is such a small percentage of the data, we can drop these rows without losing much information.
df.dropna(inplace=True)
missing = df.isnull().sum() # Here df is the DataFrame we are working with
print(missing[missing > 0])
if missing.sum() == 0:
    print("\nAll missing values have been handled.")

Series([], dtype: int64)

All missing values have been handled.


# Corelation of the Data

In [11]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]
print(df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].head())

   rooms_per_household  bedrooms_per_room  population_per_household
0             6.984127           0.146591                  2.555556
1             6.238137           0.155797                  2.109842
2             8.288136           0.129516                  2.802260
3             5.817352           0.184458                  2.547945
4             6.281853           0.172096                  2.181467


In [12]:
# Avoid division-by-zero issues (rare, but good practice)
df["rooms_per_household"] = df["total_rooms"] / df["households"].replace(0, np.nan)
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"].replace(0, np.nan)
df["population_per_household"] = df["population"] / df["households"].replace(0, np.nan)

# If any NaNs were created due to 0 denominator, drop or fill them
df = df.dropna()

In [ ]:
corr_with_target = df.select_dtypes(include="number").corr()["median_house_value"].sort_values(ascending=False)
print(corr_with_target)


median_house_value          1.000000
median_income               0.688355
rooms_per_household         0.151344
total_rooms                 0.133294
housing_median_age          0.106432
households                  0.064894
total_bedrooms              0.049686
population_per_household   -0.023639
population                 -0.025300
longitude                  -0.045398
latitude                   -0.144638
bedrooms_per_room          -0.255880
Name: median_house_value, dtype: float64


# Spliting the Data-Set

In [15]:
# Spliting the data into training and testing sets
target = "median_house_value"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
# identify numeric and categorical columns
numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = X_train.select_dtypes(exclude="number").columns  # should include ocean_proximity
print("Numeric features:", numeric_features.tolist())
print("Categorical features:", categorical_features.tolist())

Numeric features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
Categorical features: ['ocean_proximity']


# Preprocess blocks

*Numeric* 
**Meaning:**

If a numeric value is missing, fill it with the median (middle value)
Then scale numbers so they’re roughly comparable
Simple example of scaling:

income might be like 0–15
total_rooms might be like 2–39,000
Those are very different sizes. Scaling brings them into a similar range so the model behaves better.

*Categorical*
**Meaning:**

If ocean_proximity is missing, fill with the most common category
Turn categories into 0/1 columns
If the test set has a category not seen in training, don’t crash (“ignore”)

*And then Combine these*
**Meaning:**

Apply “numeric steps” to numeric columns
Apply “category steps” to categorical columns
Merge results into one clean numeric dataset for the model
Think of it like:

“Auto-clean my data before learning.”

In [ ]:
numeric_transformer = Pipeline(steps=[ # Numeric pipeline
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[ # Categorical pipeline
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Model Training and Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Evaluation Metrics are being defined here so that we can use them for all the models we will train.

def eval_model(model, X_test, y_test, name="model"):
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    print(f"{name} -> RMSE: {rmse:.2f} | MAE: {mae:.2f}")

# Base Line Regression Model
A **baseline regression model** is the **first, simplest “starting point” model** you build to predict a number (like house price).

### What it is (plain English)
- It’s a model you use as a **reference score**.
- It answers: **“If I do something simple, how good can I get?”**
- Later, when you try better models, you compare them to the baseline to see if you *actually improved*.

### In My notebook
My baseline was **Linear Regression**:
- It tries to predict `median_house_value` using a simple “more of X usually means more (or less) of Y” relationship.
- Example idea: “As median income goes up, house value usually goes up.”

### A simple example (numbers)
Imagine a tiny dataset:

| income | house_price |
|-------:|------------:|
| 2      | 100k        |
| 4      | 200k        |
| 6      | 300k        |

A baseline linear regression might learn something like:
> house_price ≈ 50k * income

It won’t be perfect in real life, but it gives you a **solid starting benchmark**.

### Why it’s useful
- It’s fast to train.
- Easy to understand.
- Tells you whether your dataset already has a strong signal.
- If a “fancier” model barely beats the baseline, then the fancy work may not be worth it.

### Where baseline regression can be used
Any time you need a quick first prediction of a number, for example:

1) **Real estate:** predict house price from size, rooms, location  
2) **Sales:** predict next month revenue from ads spend + past sales  
3) **Delivery/ride times:** predict ETA from distance + traffic level  
4) **Health:** predict blood pressure from age + weight + activity  
5) **Education:** predict exam score from study hours + attendance  

In [24]:
# Basline model using Linear Regression
from sklearn.linear_model import LinearRegression

lin = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

lin.fit(X_train, y_train)
eval_model(lin, X_test, y_test, "Linear")

Linear -> RMSE: 69042.80 | MAE: 49774.51


# Ridge Regression
I use **Ridge Regression** when I want a prediction model that’s still simple like linear regression, **but I want it to be more controlled and less “wild”**.

### What I’m trying to fix
With normal linear regression, I can accidentally make the model rely **too heavily** on some inputs (features) just to fit the training data really well. That can make my predictions less trustworthy on new data, especially when:
- some features are very similar to each other (they overlap), or  
- the dataset has some noise.

### What I do differently in Ridge
I add a rule for myself:

> “I’m allowed to use all the features, but I should keep their influence (weights) from becoming too extreme.”

So I still predict using all features, but I keep the model **more balanced**.

### The “alpha” idea (in plain English)
I choose how strict I want to be using **alpha**:

- **Small alpha** → I behave almost like normal linear regression.  
- **Big alpha** → I force the model to keep feature influence smaller and smoother.

So alpha is basically my **“calm down” slider**.

### Simple example
If I’m predicting house price and my model tries to do something extreme like:

> “Income matters *a huge amount*, rooms matter *a huge amount*, and other things barely matter…”

Ridge makes me pull that back and say:

> “Use all the features, but don’t make any single feature dominate unless it’s clearly justified.”

### Where I’d use Ridge
I use Ridge when I have:
- **lots of features**, or  
- features that overlap (like several “size-related” columns), and I want predictions that are **more stable** on new/unseen data.

If you want, I can do **Lasso Regression** next in the exact same format.

In [25]:
# Ridge (tune alpha)
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge())
])

ridge_params = {"model__alpha": [0.01, 0.1, 1, 10, 100, 300]}

ridge_search = GridSearchCV(
    ridge, ridge_params, cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)
print("Best Ridge alpha:", ridge_search.best_params_["model__alpha"])
print("Best Ridge CV RMSE:", -ridge_search.best_score_)

eval_model(ridge_search.best_estimator_, X_test, y_test, "Ridge(best)")

Best Ridge alpha: 0.1
Best Ridge CV RMSE: 68031.97008201583
Ridge(best) -> RMSE: 69036.69 | MAE: 49772.00


# Lasso Regression
I use **Lasso Regression** when I want a model like linear regression, **but I also want it to automatically simplify itself** by removing features that don’t help much.

### What I’m trying to fix
Sometimes I have many features, and some are:
- weak,
- noisy,
- or basically repeating what other features already say.

If I keep all of them, the model can become harder to interpret and may not generalize as well.

### What I do differently in Lasso
I set a rule for myself:

> “Try to predict well, but if a feature isn’t truly useful, set its influence to exactly zero.”

So unlike Ridge (which usually just *shrinks* feature influence), Lasso can **completely drop** some features.

### The “alpha” idea (in plain English)
**alpha** controls how aggressively I simplify:

- **Small alpha** → I keep most features (close to normal linear regression).  
- **Big alpha** → I drop more features (more influences become exactly 0).

So alpha is my **“how strict should I be about simplicity?”** slider.

### Simple example
Imagine I’m predicting house price using:

- income (very useful)
- location (useful)
- total_rooms (maybe useful)
- a weak feature (barely related)

Lasso might decide:
- income: keep it  
- location: keep it  
- total_rooms: keep it a little  
- weak feature: **set its influence to 0** (basically “ignore it”)

That makes the model simpler and easier to explain.

### Where I’d use Lasso
I use Lasso when I want:
- a model that’s easier to explain (“these are the few features that mattered”), or
- a model that automatically reduces clutter when I have many features.

If you want, I can explain **Polynomial Regression** next in the same format (since that was your best-performing model).

In [26]:
# Lasso (tune alpha)
from sklearn.linear_model import Lasso

lasso = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Lasso(max_iter=50000))
])

lasso_params = {"model__alpha": [0.0001, 0.001, 0.01, 0.1, 1]}

lasso_search = GridSearchCV(
    lasso, lasso_params, cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

lasso_search.fit(X_train, y_train)
print("Best Lasso alpha:", lasso_search.best_params_["model__alpha"])
print("Best Lasso CV RMSE:", -lasso_search.best_score_)

eval_model(lasso_search.best_estimator_, X_test, y_test, "Lasso(best)")

Best Lasso alpha: 1
Best Lasso CV RMSE: 68032.08089392023
Lasso(best) -> RMSE: 69036.81 | MAE: 49771.94


# Polynomial Regression
I use **Polynomial Regression** when a straight-line relationship is too simple, and I need my model to **capture curved patterns** in the data.

### What I’m trying to fix
Normal linear regression basically assumes:

> “If X increases by 1, Y changes by the same amount every time.”

But real life often doesn’t work like that. Sometimes:
- prices rise slowly at first, then faster,
- or a feature helps up to a point, then stops helping,
- or two features together create an effect that neither has alone.

A straight line can’t describe those patterns well.

### What I do differently in Polynomial Regression
I create extra “curved” versions of the same features so the model can learn curved relationships.

That means I let the model use things like:
- **X²** (squared terms) to create curves  
- **X × Y** (interaction terms) to capture “combo effects”

Important: the model is still doing prediction using a formula, but now that formula has **more shape** than a straight line.

### The “degree” idea (in plain English)
The **degree** controls how flexible the curve can be:

- **degree = 1** → this is just normal linear regression (straight line)
- **degree = 2** → allows simple curves (often the sweet spot)
- **degree = 3+** → more complex curves (can fit more patterns, but can also become messy)

So degree is my **“how curvy should the model be?”** setting.

### Simple example
Suppose house price vs income behaves like this:

- income from 1 → 3: price increases slowly  
- income from 3 → 8: price increases much faster

A straight line struggles here.

With **degree = 2**, the model can learn a curve like:

> price ≈ a(income) + b(income²)

That curve can bend upward and fit the pattern better.

### Where I’d use Polynomial Regression
I use polynomial regression when I suspect the relationship is **not straight**, for example:

- **house prices** (income/location effects are often not perfectly linear)
- **marketing spend vs sales** (returns may accelerate or plateau)
- **learning hours vs exam score** (improvement may slow after many hours)
- **speed vs fuel consumption** (often curved)
- **dose vs effect** in health/biology (rarely a straight line)

If you want, I can explain why Polynomial Regression often works best **only when combined with Ridge** (like you did), in the same exact format.

In [27]:
# Polynominal Regression (tune degree)
from sklearn.preprocessing import PolynomialFeatures

numeric_poly_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

preprocess_poly = ColumnTransformer(
    transformers=[
        ("num", numeric_poly_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

poly_ridge = Pipeline(steps=[
    ("preprocess", preprocess_poly),
    ("model", Ridge(alpha=10))
])

poly_ridge.fit(X_train, y_train)
eval_model(poly_ridge, X_test, y_test, "Poly(deg=2)+Ridge(alpha=10)")

Poly(deg=2)+Ridge(alpha=10) -> RMSE: 65212.81 | MAE: 46612.52


In [36]:
# Now We will get the Model with the best performance and we will save it using joblib so that we can use it later without retraining.
import joblib
best_model = poly_ridge
joblib.dump(best_model, "best_california_housing_model.joblib")
print("Best model saved as 'best_california_housing_model.joblib'")

Best model saved as 'best_california_housing_model.joblib'
